---

Building `Stateful` Assistants with Tools: A Hands-On Introduction

- Overview
    - Assistant
    - threads
    - messages
    - runs
    - tools
    - steps

---

In [1]:
import os
from openai import OpenAI

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
client = OpenAI()

In [3]:
import json
from IPython.display import display

def show_json(obj):
    """
    Pretty prints a JSON representation of a Pydantic object or dict-like object.
    """
    try:
        # Try model_dump_json (for Pydantic or OpenAI objects)
        json_data = obj.model_dump() if hasattr(obj, "model_dump") else obj
        
    except AttributeError:
        # Fallback to dict or raw object
        json_data = obj

    # Pretty print
    display(json.loads(json.dumps(json_data, indent=2, default=str)))

#### Assistants
---

- Allows you to create an “assistant” object with:
    - A `system prompt` (instructions)
    - `Model` choice (e.g., GPT-4)
    - `Tools` (like code interpreter, retrieval, etc.)
    - `File uploads` for context

- Managed long-running conversations with `memory`, `threads`, and `tool usage`.


| Component      | Role                                                                         |
| -------------- | ---------------------------------------------------------------------------- |
| **Assistant**  | Blueprint of your AI agent (personality, tools, instructions).               |
| **Thread**     | Conversation instance where all messages (user/assistant) are kept.          |
| **Run**        | Execution of a specific query within a `thread` using an `assistant`.            |
| `Tool Calls` | Allowed the assistant to call tools like Python code execution or retrieval. |


In [4]:
assistant = client.beta.assistants.create(
    name        = "Math Tutor",
    instructions= "You are a personal math tutor. Answer questions briefly, in a sentence or less.",
    model       = "gpt-4o",
)

show_json(assistant)

{'id': 'asst_UtpbBtF5KcFBWBAaKkjiqd7u',
 'created_at': 1759230045,
 'description': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': None, 'file_search': None},
 'top_p': 1.0,
 'reasoning_effort': None}

- You can create an Assistant either via the OpenAI Dashboard or using the API.
- After creating the Assistant, note down the `Assistant ID` — it's essential for future interactions.
- Next step: create a new Thread to manage the conversation.
- Add a Message to the Thread to start the conversation.
- Threads maintain conversation state, so you don’t need to re-send the full message history each time.

#### Threads
---

In OpenAI’s Assistants API, a thread is essentially a `conversation container` that stores the history of messages exchanged between you and an assistant.

Think of it as a chat session ID that you can use to continue conversations without losing context.

Purpose

- A thread maintains `state` across multiple API calls.
- You don’t have to pass the entire conversation every time.
- It works like a "chat room" where all messages are logged.

In [5]:
thread = client.beta.threads.create()

show_json(thread)

{'id': 'thread_BoBfHzjaV32u1w03GU2OWuD2',
 'created_at': 1759230089,
 'metadata': {},
 'object': 'thread',
 'tool_resources': {'code_interpreter': None, 'file_search': None}}

In [6]:
messages = client.beta.threads.messages.list(thread_id=thread.id)

for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

**OpenAI Assistant Tool Resources**


`code_interpreter`
- Refers to the **Python Code Interpreter** (also known as **Advanced Data Analysis**).
- When enabled, your assistant can:
  - Write and execute Python code
  - Analyze CSV and other data files
  - Perform calculations and create data visualizations
- If it shows `None`, it means this tool is **not configured** for the current assistant.

---

`file_search`
- Used for **retrieval-based queries**, allowing the assistant to:
  - Search through uploaded files
  - Access a linked **knowledge base** or **vector store**
- Typically, you must **attach a vector store or files** to enable this functionality.
- If it shows `None`, the assistant has **no access** to files or a retrieval tool.

---


**add the Message to the thread:**

In [7]:
message = client.beta.threads.messages.create(
    thread_id = thread.id,
    role      = "user",
    content   = "I need to solve the equation `3x + 11 = 14`. Can you help me?",
)

show_json(message)

{'id': 'msg_mj7PBDkkIU9BisFlSCpgTIVl',
 'assistant_id': None,
 'attachments': [],
 'completed_at': None,
 'content': [{'text': {'annotations': [],
    'value': 'I need to solve the equation `3x + 11 = 14`. Can you help me?'},
   'type': 'text'}],
 'created_at': 1759230146,
 'incomplete_at': None,
 'incomplete_details': None,
 'metadata': {},
 'object': 'thread.message',
 'role': 'user',
 'run_id': None,
 'status': None,
 'thread_id': 'thread_BoBfHzjaV32u1w03GU2OWuD2'}

In [8]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

user: I need to solve the equation `3x + 11 = 14`. Can you help me?


#### Runs
---
- Understanding Runs in the Assistants API
    - `Threads` are independent of `Assistants`
        - Unlike ChatGPT, where a thread is tied to a model, `Assistants API` decouples `Threads` and `Assistants`.

    - You can create a `Thread` first and later associate it with `any Assistant` using a Run.

- What is a `Run`?
    - A Run connects an `Assistant` to a `Thread` so it can read messages and generate responses.
    - `Runs` trigger the `Assistant` to act:
        - Read existing messages in the thread.
        - Generate a reply (or replies).
        - Possibly use tools like `code interpreter`, `retrieval`, etc.

In [9]:
run = client.beta.threads.runs.create(
    thread_id   = thread.id,
    assistant_id= assistant.id,
)

show_json(run)

{'id': 'run_3W5MZ1hoK4RVflXKaw4u4Xpb',
 'assistant_id': 'asst_UtpbBtF5KcFBWBAaKkjiqd7u',
 'cancelled_at': None,
 'completed_at': None,
 'created_at': 1759230375,
 'expires_at': 1759230975,
 'failed_at': None,
 'incomplete_details': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'thread.run',
 'parallel_tool_calls': True,
 'required_action': None,
 'response_format': 'auto',
 'started_at': None,
 'status': 'queued',
 'thread_id': 'thread_BoBfHzjaV32u1w03GU2OWuD2',
 'tool_choice': 'auto',
 'tools': [],
 'truncation_strategy': {'type': 'auto', 'last_messages': None},
 'usage': None,
 'temperature': 1.0,
 'top_p': 1.0,
 'tool_resources': {},
 'reasoning_effort': None}

In [10]:
run.status

'queued'

In [11]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

assistant: Subtract 11 from both sides to get \(3x = 3\), then divide by 3 to find \(x = 1\).
user: I need to solve the equation `3x + 11 = 14`. Can you help me?


#### Run is Asynchronous in the Assistants API
---

**Creating a Run is not immediate like Chat Completions API:**

- When you create a Run, it **returns instantly** with **metadata**.
- The metadata includes the **status**, which starts as `"queued"`.

**Assistant processes the Run in the background:**

- The Assistant can **use tools**, **generate messages**, or perform other operations.
- The **status evolves**: 
    - `"queued"` → Waiting to start.
    - `"in_progress"` → Assistant is working.
    - `"completed"` → Done processing.
    - `"requires_action"` → Waiting on a `tool call` or `user input`.

**You must poll to track progress:**

- Use a **loop** to keep checking the Run status.

**Advanced insight with `Run Steps:`**

- Each Run can have **multiple intermediate steps**.
- These steps allow **deeper visibility** into what the Assistant is doing

**Note:**
- **Streaming support** (real-time updates) - is not yet available in the Assistants API.


In [12]:
import time

In [14]:
def wait_on_run(run, thread):
    while run.status == "queued" or run.status == "in_progress":
        run = client.beta.threads.runs.retrieve(
            thread_id = thread.id,
            run_id    = run.id,
        )
        time.sleep(0.5)
    return run

In [15]:
run = wait_on_run(run, thread)
show_json(run)

{'id': 'run_3W5MZ1hoK4RVflXKaw4u4Xpb',
 'assistant_id': 'asst_UtpbBtF5KcFBWBAaKkjiqd7u',
 'cancelled_at': None,
 'completed_at': 1759230378,
 'created_at': 1759230375,
 'expires_at': None,
 'failed_at': None,
 'incomplete_details': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'last_error': None,
 'max_completion_tokens': None,
 'max_prompt_tokens': None,
 'metadata': {},
 'model': 'gpt-4o',
 'object': 'thread.run',
 'parallel_tool_calls': True,
 'required_action': None,
 'response_format': 'auto',
 'started_at': 1759230376,
 'status': 'completed',
 'thread_id': 'thread_BoBfHzjaV32u1w03GU2OWuD2',
 'tool_choice': 'auto',
 'tools': [],
 'truncation_strategy': {'type': 'auto', 'last_messages': None},
 'usage': {'completion_tokens': 32,
  'prompt_tokens': 66,
  'total_tokens': 98,
  'prompt_token_details': {'cached_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 0}},
 'temperature': 1.0,
 'top_p': 1.0,
 'tool_res

#### Messages
---

Now that the `Run` has completed, we can list the `Messages` in the `Thread` to see what got added by the `Assistant`.

In [16]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
show_json(messages)

{'data': [{'id': 'msg_Yci8NvJ5UxWrr3DN0SxPFSgP',
   'assistant_id': 'asst_UtpbBtF5KcFBWBAaKkjiqd7u',
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [],
      'value': 'Subtract 11 from both sides to get \\(3x = 3\\), then divide by 3 to find \\(x = 1\\).'},
     'type': 'text'}],
   'created_at': 1759230377,
   'incomplete_at': None,
   'incomplete_details': None,
   'metadata': {},
   'object': 'thread.message',
   'role': 'assistant',
   'run_id': 'run_3W5MZ1hoK4RVflXKaw4u4Xpb',
   'status': None,
   'thread_id': 'thread_BoBfHzjaV32u1w03GU2OWuD2'},
  {'id': 'msg_mj7PBDkkIU9BisFlSCpgTIVl',
   'assistant_id': None,
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [],
      'value': 'I need to solve the equation `3x + 11 = 14`. Can you help me?'},
     'type': 'text'}],
   'created_at': 1759230146,
   'incomplete_at': None,
   'incomplete_details': None,
   'metadata': {},
   'object': 'thread.message',
   'ro

####  Messages Are Ordered in Reverse-Chronological Order
---

- When retrieving messages from a **Run**, the results are **ordered in reverse-chronological order**.
- This means the **most recent messages come first**.

**Why this matters:**
- This is **different from the Chat Completions API**, where messages are typically shown **from oldest to newest** (i.e., chronological order).
- Reverse order helps ensure the latest Assistant response or tool action appears on the **first page** of results, especially when paginating.

**Keep in mind:**
- When processing or displaying the conversation, you might need to **reverse the order** to get a natural flow (oldest → newest).

In [17]:
# Create a message to append to our thread
# sends a user message to an existing thread in the OpenAI Assistants API
message = client.beta.threads.messages.create(
    thread_id = thread.id, 
    role      = "user", 
    content   = "Could you explain this to me?"
)

In [18]:
# Execute our run
run = client.beta.threads.runs.create(
    thread_id   = thread.id,
    assistant_id= assistant.id,
)

In [19]:
# Wait for completion
wait_on_run(run, thread)

Run(id='run_cwas870hgZTz1KzMoYfGwuAM', assistant_id='asst_UtpbBtF5KcFBWBAaKkjiqd7u', cancelled_at=None, completed_at=1759230586, created_at=1759230583, expires_at=None, failed_at=None, incomplete_details=None, instructions='You are a personal math tutor. Answer questions briefly, in a sentence or less.', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4o', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=1759230584, status='completed', thread_id='thread_BoBfHzjaV32u1w03GU2OWuD2', tool_choice='auto', tools=[], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usage=Usage(completion_tokens=67, prompt_tokens=111, total_tokens=178, prompt_token_details={'cached_tokens': 0}, completion_tokens_details={'reasoning_tokens': 0}), temperature=1.0, top_p=1.0, tool_resources={}, reasoning_effort=None)

In [20]:
# Retrieve all the messages added after our last user message
messages = client.beta.threads.messages.list(
    thread_id= thread.id, 
    order    = "asc", 
    after    = message.id
)

show_json(messages)

# ✅ You will only see assistant responses or tool calls made after that message.
# ❌ You will not see previous user messages or earlier conversation history because of the after=message.id filter.

{'data': [{'id': 'msg_NE2NttbBlGm3OzrI6g74OI0d',
   'assistant_id': 'asst_UtpbBtF5KcFBWBAaKkjiqd7u',
   'attachments': [],
   'completed_at': None,
   'content': [{'text': {'annotations': [],
      'value': 'Certainly! Start by subtracting 11 from both sides of the equation to isolate \\(3x: 3x = 14 - 11\\). This simplifies to \\(3x = 3\\). Next, divide both sides by 3 to solve for \\(x\\), getting \\(x = 1\\).'},
     'type': 'text'}],
   'created_at': 1759230585,
   'incomplete_at': None,
   'incomplete_details': None,
   'metadata': {},
   'object': 'thread.message',
   'role': 'assistant',
   'run_id': 'run_cwas870hgZTz1KzMoYfGwuAM',
   'status': None,
   'thread_id': 'thread_BoBfHzjaV32u1w03GU2OWuD2'}],
 'has_more': False,
 'object': 'list',
 'first_id': 'msg_NE2NttbBlGm3OzrI6g74OI0d',
 'last_id': 'msg_NE2NttbBlGm3OzrI6g74OI0d'}

#### Putting them together

In [21]:
MATH_ASSISTANT_ID = assistant.id  # or a hard-coded ID like "asst-..."
MATH_ASSISTANT_ID

'asst_UtpbBtF5KcFBWBAaKkjiqd7u'

In [22]:
client = OpenAI()

In [23]:
def submit_message(assistant_id, thread, user_message):
    
    client.beta.threads.messages.create(
            thread_id = thread.id, 
            role      = "user", 
            content   = user_message
    )
    
    return client.beta.threads.runs.create(
                thread_id   = thread.id,
                assistant_id= assistant_id,
           )

**Purpose:**

- Submits a new user message to an existing thread and initiates a Run using a specific Assistant.

**Steps:**
- Add user message to the thread:
    - Uses client.beta.threads.messages.create(...).
    - `role` = "user" specifies this message is from the user.
    - `content` is the user's input message.

**Create a Run:**

- Triggers the Assistant to process the newly added message in that thread.
- Returns the Run object, which can later be polled for status (queued, in_progress, completed, etc.).

In [24]:
def get_response(thread):
    return client.beta.threads.messages.list(thread_id = thread.id, 
                                             order     = "asc")

**Purpose:**
  
- Retrieves the full list of messages in the thread, including assistant responses.

**Details:**
- Messages are returned in chronological order (order="asc") so that the conversation flows from oldest to newest.
- You can filter or inspect messages (e.g., checking role == "assistant" to get the reply).

In [25]:
def create_thread_and_run(user_input):
    thread = client.beta.threads.create()
    run    = submit_message(MATH_ASSISTANT_ID, thread, user_input)
    return thread, run

**Purpose:**
- Creates a new conversation thread and sends the first user message to it.
- Kicks off a Run so that the Assistant processes this first message.

**Steps:**
- Create a thread:
    - Threads are containers to store messages and maintain conversational state.

- Submit user message and trigger Run:
    - Calls submit_message(...) to handle message submission and start the processing run.

- Returns:
    - The `thread` object (used for further interactions).
    - The `run` object (used to track processing progress).

In [26]:
# Emulating concurrent user requests
thread1, run1 = create_thread_and_run(
                    "I need to solve the equation `3x + 11 = 14`. Can you help me?"
                )

thread2, run2 = create_thread_and_run(
                    "Could you explain linear algebra to me?"
                )

thread3, run3 = create_thread_and_run(
                    "I don't like math. What can I do?"
                )

Once all Runs are going, we can wait on each and get the responses.

In [27]:
import time

In [30]:
# Pretty printing helper
def pretty_print(messages):
    print("# Messages")
    for m in messages:
        print(f"{m.role}: {m.content[0].text.value}")
    print()

In [31]:
# Waiting in a loop
def wait_on_run(run, thread):
    while run.status == "queued" or run.status == "in_progress":
        run = client.beta.threads.runs.retrieve(
            thread_id = thread.id,
            run_id    = run.id,
        )
        time.sleep(0.5)
    return run

In [32]:
# Wait for Run 1
run1 = wait_on_run(run1, thread1)
pretty_print(get_response(thread1))

# Wait for Run 2
run2 = wait_on_run(run2, thread2)
pretty_print(get_response(thread2))

# Wait for Run 3
run3 = wait_on_run(run3, thread3)
pretty_print(get_response(thread3))

# Messages
user: I need to solve the equation `3x + 11 = 14`. Can you help me?
assistant: Sure! Subtract 11 from both sides to get \(3x = 3\), then divide by 3 to find \(x = 1\).

# Messages
user: Could you explain linear algebra to me?
assistant: Linear algebra is the branch of mathematics that deals with vectors, vector spaces, and linear transformations, often represented using matrices.

# Messages
user: I don't like math. What can I do?
assistant: Try finding real-world applications of math that interest you, like budgeting, cooking, or sports statistics, to make it more engaging and relevant.



In [33]:
# Thank our assistant on Thread 3 :)
run4 = submit_message(MATH_ASSISTANT_ID, thread3, "Thank you!")
run4 = wait_on_run(run4, thread3)
pretty_print(get_response(thread3))

# Messages
user: I don't like math. What can I do?
assistant: Try finding real-world applications of math that interest you, like budgeting, cooking, or sports statistics, to make it more engaging and relevant.
user: Thank you!
assistant: You're welcome! If you have any more questions or need help, feel free to ask.



---
#### A simple chatbot
---

Turn 1

In [29]:
assistant = client.beta.assistants.create(
    name         ="Simple Chatbot",
    instructions ="You are a helpful chatbot that gives short, clear answers.",
    model        ="gpt-4o",
)

print("Assistant ID:", assistant.id)

thread = client.beta.threads.create()
print("Thread ID:", thread.id)

message = client.beta.threads.messages.create(
    thread_id = thread.id,
    role      = "user",
    content   = "What is the capital of India"
)

run = client.beta.threads.runs.create(
    thread_id   =thread.id,
    assistant_id=assistant.id
)

# Wait for run to complete
while True:
    run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    if run_status.status in ["completed", "failed", "cancelled"]:
        break
    time.sleep(.25)

# List all messages
messages = client.beta.threads.messages.list(thread_id=thread.id, order="asc")

for msg in messages.data:
    print(f"{msg.role.upper()}: {msg.content[0].text.value}")

Assistant ID: asst_9gpJn9WnUCj58CqivE5YjZUM
Thread ID: thread_w0SHwIPatPab4OO7b5GnxXMQ
USER: What is the capital of India
ASSISTANT: The capital of India is New Delhi.


Turn 2

In [30]:
message = client.beta.threads.messages.create(
    thread_id = thread.id,
    role      = "user",
    content   = "What about China"
)

run = client.beta.threads.runs.create(
    thread_id   =thread.id,
    assistant_id=assistant.id
)

# Wait for run to complete
while True:
    run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    if run_status.status in ["completed", "failed", "cancelled"]:
        break
    time.sleep(.25)

# List all messages
messages = client.beta.threads.messages.list(thread_id=thread.id, order="asc")

for msg in messages.data:
    print(f"{msg.role.upper()}: {msg.content[0].text.value}")

USER: What is the capital of India
ASSISTANT: The capital of India is New Delhi.
USER: What about China
ASSISTANT: The capital of China is Beijing.


Turn 3

In [31]:
message = client.beta.threads.messages.create(
    thread_id = thread.id,
    role      = "user",
    content   = "Compare geographical areas"
)

run = client.beta.threads.runs.create(
    thread_id   = thread.id,
    assistant_id= assistant.id
)

# Wait for run to complete
while True:
    run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    if run_status.status in ["completed", "failed", "cancelled"]:
        break
    time.sleep(.25)

# List all messages
messages = client.beta.threads.messages.list(thread_id=thread.id, order="asc")

for msg in messages.data:
    print(f"{msg.role.upper()}: {msg.content[0].text.value}")

USER: What is the capital of India
ASSISTANT: The capital of India is New Delhi.
USER: What about China
ASSISTANT: The capital of China is Beijing.
USER: Compare geographical areas
ASSISTANT: China is geographically larger than India. China's total area is approximately 9.6 million square kilometers, making it the fourth largest country in the world by area. India, on the other hand, has a total area of about 3.3 million square kilometers.


#### memory
----
  
`Context window` is still limited by the model (e.g., ~128k tokens for GPT-4o).

Very long threads may have older messages summarized or dropped automatically to fit context.

---
#### Chatbot with persistent memory
---

In [32]:
# ---------- Persistent Memory Storage ----------
MEMORY_FILE = "chat_memory.json"


In [35]:
# Load saved memory (thread_id) if exists
# try:
#     with open(MEMORY_FILE, "r") as f:
#         memory = json.load(f)
# except FileNotFoundError:
#     memory = {}
    
try:
    with open(MEMORY_FILE, "r") as f:
        memory = json.load(f)
        if not isinstance(memory, dict):
            memory = {}
except FileNotFoundError:
    memory = {}

In [36]:
# Create Assistant (replace with your assistant_id if already exists)
assistant = client.beta.assistants.create(
    name        ="Memory Chatbot",
    instructions="You are a helpful assistant with memory. Remember the conversation context.",
    model       ="gpt-4o"
)

# Retrieve or create thread
thread_id = memory.get("thread_id")

if not thread_id:
    thread    = client.beta.threads.create()
    thread_id = thread.id
    
    memory["thread_id"] = thread_id
    
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f)

print("Using Thread:", thread_id)

Using Thread: thread_qaFTXfnxqmzWsKzXWoU0ckIX


In [ ]:
# ---------- Chat Loop ----------
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye! Memory saved for next session.")
        break

    # Add user message
    client.beta.threads.messages.create(
        thread_id= thread_id,
        role     = "user",
        content  = user_input
    )

    # Create run
    run = client.beta.threads.runs.create(
        thread_id   = thread_id,
        assistant_id= assistant.id
    )

    # Wait for completion
    while True:
        run_status = client.beta.threads.runs.retrieve(thread_id=thread_id, run_id=run.id)
        if run_status.status == "completed":
            break

    # Get assistant response
    messages = client.beta.threads.messages.list(thread_id=thread_id, order="desc", limit=1)
    reply    = messages.data[0].content[0].text.value
    
    print("Bot:", reply)

In [109]:
# Persist after exiting the loop
with open(MEMORY_FILE, "w") as f:
    json.dump(memory, f)

---
#### Tools
----

A key feature of the Assistants API is the ability to equip our Assistants with Tools, like `Code Interpreter`, `File Search`, and `custom Functions`.

**Code Interpreter**

In [110]:
assistant = client.beta.assistants.update(
    MATH_ASSISTANT_ID,
    tools =[{"type": "code_interpreter"}],
)

show_json(assistant)

{'id': 'asst_dYf7tSwg0E4S0E3g5fuIf87v',
 'created_at': 1754333557,
 'description': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [{'type': 'code_interpreter'}],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': {'file_ids': []}, 'file_search': None},
 'top_p': 1.0,
 'reasoning_effort': None}

Now, let's ask the Assistant to use its new tool.

In [111]:
thread, run = create_thread_and_run(
                    "Generate the first 20 fibbonaci numbers with code."
                )

run = wait_on_run(run, thread)

pretty_print(get_response(thread))

# Messages
user: Generate the first 20 fibbonaci numbers with code.
assistant: The first 20 Fibonacci numbers are: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181.



#### What Are Steps in a Run?

- A Run is the process triggered after the user sends a message.
- A Step is a single action or decision taken by the Assistant within that Run.

- Each Run contains multiple Steps that show what the Assistant did, such as:

    - Thinking (writing)
    - Calling a tool (like code_interpreter, retrieval, or function)
    - Returning output

**Why Steps Matter**

- Track progress of the Assistant’s thought process
- Debug tool use (e.g., did the Assistant use code_interpreter?)
- Surface UI state, like showing a spinner if code is being written
- Trace how the final answer was constructed


In [112]:
run_steps = client.beta.threads.runs.steps.list(
            thread_id = thread.id, 
            run_id    = run.id, 
            order     = "asc"
            )

Let's take a look at each Step's step_details.

In [113]:
for step in run_steps.data:
    step_details = step.step_details
    
    print(json.dumps(show_json(step_details), indent=4))

{'tool_calls': [{'id': 'call_HiwdFJFH2fHMEIiuVQ6w62tl',
   'code_interpreter': {'input': 'def generate_fibonacci(n):\r\n    fibonacci_numbers = [0, 1]\r\n    for i in range(2, n):\r\n        next_number = fibonacci_numbers[-1] + fibonacci_numbers[-2]\r\n        fibonacci_numbers.append(next_number)\r\n    return fibonacci_numbers[:n]\r\n\r\n# Generate the first 20 Fibonacci numbers.\r\nfirst_20_fibonacci = generate_fibonacci(20)\r\nfirst_20_fibonacci',
    'outputs': []},
   'type': 'code_interpreter'}],
 'type': 'tool_calls'}

null


{'message_creation': {'message_id': 'msg_OMsJvZ2K2EiKjNOmmYXvduPR'},
 'type': 'message_creation'}

null


**Step 1: tool_calls**

- "type": "tool_calls" indicates this is a tool execution step.
- "code_interpreter" confirms the assistant used the Code Interpreter.
- "input" shows the generated Python code.
- "outputs" contains the results (e.g., printed output, plots).

**Step 2: message_creation**

- This represents the assistant posting the final message back to the user thread.

---
#### File search
---
Another powerful tool in the Assistants API is File search. 

This allows the uploading of files to the Assistant to be used as a knowledge base when answering questions.

In [114]:
# Upload the file
file = client.files.create(
    file=open(
        "language_models_are_unsupervised_multitask_learners.pdf",
        "rb",
    ),
    purpose="assistants",
)

In [115]:
# Create a vector store
vector_store = client.vector_stores.create(
    name="language_models_are_unsupervised_multitask_learners",
)

In [116]:
# Add the file to the vector store
vector_store_file = client.vector_stores.files.create_and_poll(
                    vector_store_id = vector_store.id,
                    file_id         = file.id,
)

In [117]:
# Confirm the file was added
while vector_store_file.status == "in_progress":
    time.sleep(1)
if vector_store_file.status == "completed":
    print("File added to vector store")
elif vector_store_file.status == "failed":
    raise Exception("Failed to add file to vector store")

File added to vector store


In [118]:
# Update Assistant
assistant = client.beta.assistants.update(
    MATH_ASSISTANT_ID,
    tools=[{"type": "code_interpreter"}, {"type": "file_search"}],
    tool_resources={
        "file_search":{
            "vector_store_ids": [vector_store.id]
        },
        "code_interpreter": {
            "file_ids": [file.id]
        }
    },
)

In [119]:
show_json(assistant)

{'id': 'asst_dYf7tSwg0E4S0E3g5fuIf87v',
 'created_at': 1754333557,
 'description': None,
 'instructions': 'You are a personal math tutor. Answer questions briefly, in a sentence or less.',
 'metadata': {},
 'model': 'gpt-4o',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [{'type': 'code_interpreter'},
  {'type': 'file_search',
   'file_search': {'max_num_results': None,
    'ranking_options': {'score_threshold': 0.0,
     'ranker': 'default_2024_08_21'}}}],
 'response_format': 'auto',
 'temperature': 1.0,
 'tool_resources': {'code_interpreter': {'file_ids': ['file-5X7qxJM5xHc7eV9KDSxSo6']},
  'file_search': {'vector_store_ids': ['vs_689109b108dc8191a9b62ce04395bd6b']}},
 'top_p': 1.0,
 'reasoning_effort': None}

In [120]:
thread, run = create_thread_and_run(
    "What are some cool math concepts behind this ML paper pdf? Explain in two sentences."
)

In [121]:
run = wait_on_run(run, thread)
pretty_print(get_response(thread))

# Messages
user: What are some cool math concepts behind this ML paper pdf? Explain in two sentences.
assistant: The paper explores the concept of multitask learning and meta-learning, where a model is trained to perform multiple tasks by treating input, task instructions, and output as sequences of symbols, allowing it to learn from diverse tasks without explicit supervision【6:0†source】. It also uses probabilistic frameworks to estimate conditional distributions, which helps in modeling multitask systems by employing architectures like Transformers for self-attention and optimization techniques like MAML【6:8†source】.



In [122]:
# Delete the vector store
client.vector_stores.delete(vector_store.id)

VectorStoreDeleted(id='vs_689109b108dc8191a9b62ce04395bd6b', deleted=True, object='vector_store.deleted')

---
#### Function calling
---

In [138]:
tools = [
    {
        "type": "function",  # required
        "function": {        # ✅ must nest inside this
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "default": "celsius"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [139]:
# Create assistant with tool
assistant = client.beta.assistants.create(
    name        ="WeatherBot",
    instructions="You can fetch weather via function call.",
    model       ="gpt-4-1106-preview",
    tools       =tools
)

In [140]:
thread = client.beta.threads.create()

In [141]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role     ="user",
    content  ="What's the weather in Mumbai?"
)

In [142]:
run = client.beta.threads.runs.create_and_poll(
    thread_id   =thread.id,
    assistant_id=assistant.id
)

Check if Function Call is Needed

In [143]:
if run.status == "requires_action":
    action    = run.required_action.submit_tool_outputs.tool_calls[0]
    tool_name = action.function.name
    arguments = json.loads(action.function.arguments)

    print("Tool requested:", tool_name)
    print("Arguments:", arguments)


Tool requested: get_current_weather
Arguments: {'city': 'Mumbai'}


Implement the Tool

In [144]:
def get_current_weather(city, unit="celsius"):
    # Mocked weather data
    weather_data = {
        "Mumbai": {"temp": 32, "condition": "Sunny"},
        "Delhi": {"temp": 36, "condition": "Hot and Dry"}
    }
    data     = weather_data.get(city, {"temp": 25, "condition": "Clear"})
    unit_str = "°C" if unit == "celsius" else "°F"
    
    return f"The weather in {city} is {data['temp']}{unit_str} and {data['condition']}."


Send Tool Output Back to Assistant

In [145]:
client.beta.threads.runs.submit_tool_outputs(
    thread_id   =thread.id,
    run_id      =run.id,
    tool_outputs=[{
        "tool_call_id": action.id,
        "output": get_current_weather(arguments["city"], arguments.get("unit", "celsius"))
    }]
)


Run(id='run_dVTZyEQCYFNKjytd01evIlti', assistant_id='asst_LkIDTR6u2rjYdPv230zSpMW8', cancelled_at=None, completed_at=None, created_at=1754336106, expires_at=1754336706, failed_at=None, incomplete_details=None, instructions='You can fetch weather via function call.', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4-1106-preview', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=1754336106, status='queued', thread_id='thread_Q0rdY9OuItu7983CJG6yxyUZ', tool_choice='auto', tools=[FunctionTool(function=FunctionDefinition(name='get_current_weather', description='Get the current weather in a given location', parameters={'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'City name'}, 'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit'], 'default': 'celsius'}}, 'required': ['city']}, strict=False), type='function')], truncation_strategy=TruncationStrategy(type='a

In [146]:
messages = client.beta.threads.messages.list(thread_id=thread.id, order="asc")
for msg in messages.data:
    print(f"{msg.role.upper()}: {msg.content[0].text.value}")

USER: What's the weather in Mumbai?
ASSISTANT: The current weather in Mumbai is 32°C and sunny.


---
#### HITL with Assistant API
---

In [150]:
assistant = client.beta.assistants.create(
    name        ="HITL Bot",
    instructions="Provide suggestions for responses. A human will approve or edit before finalizing.",
    model       ="gpt-4o"
)

In [151]:
thread = client.beta.threads.create()

In [152]:
client.beta.threads.messages.create(
    thread_id=thread.id,
    role     ="user",
    content  ="Write a short welcome message for a new customer."
)

Message(id='msg_ephegkSn4e1e7SDLlwfgZ7aE', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='Write a short welcome message for a new customer.'), type='text')], created_at=1754336451, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_rTmpXFuGrUozPFJ2sJqaALD3')

In [153]:
run = client.beta.threads.runs.create(
    thread_id   =thread.id,
    assistant_id=assistant.id
)

In [154]:
# Wait until the assistant finishes
while True:
    run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    if run_status.status in ["completed", "failed", "expired"]:
        break

In [155]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
draft    = messages.data[0].content[0].text.value
print("\nAI Draft Suggestion:", draft)


AI Draft Suggestion: Hello [Customer's Name],

Welcome to [Company Name]! We’re thrilled to have you with us and look forward to serving you. Whether you’re looking for [product/service] or any other needs, our team is here to help. If you have any questions or need assistance, feel free to reach out. We hope your experience with us exceeds your expectations!

Warm regards,  
[Your Name/Team Name]  
[Company Name]


In [156]:
# HITL step: human reviews
feedback = input("Accept (a), Edit (e), Exit (x): ").lower()

Accept (a), Edit (e), Exit (x):  x


In [158]:
if feedback == "e":
    edited        = input("Enter your edited message: ")
    final_message = edited
elif feedback == "x":
    print("Session ended by user.")
    final_message = None
else:
    final_message = draft

# Persist final message if accepted or edited
if final_message:
    client.beta.threads.messages.create(
        thread_id=thread.id,
        role     ="user",
        content  =f"Final approved message: {final_message}"
    )
print("\nFinal message saved:", final_message)

Session ended by user.

Final message saved: None
